In [2]:
import pandas as pd
import numpy as np
import torch
import torch.utils.data as data
import torch.nn as nn
import numpy as np
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler

In [103]:
class RMSLELoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, y_pred, y_true):
        y_pred = torch.clamp(y_pred, min=0)
        log_pred = torch.log1p(y_pred)
        log_true = torch.log1p(y_true)
        return torch.sqrt(torch.mean((log_pred - log_true) ** 2))

In [104]:
def standarized_data(X_train, X_valid, X_test):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)  # fit + transform
    X_valid_scaled = scaler.transform(X_valid)  # tylko transform
    X_test_scaled = scaler.transform(X_test)  # tylko transform

    return X_train_scaled, X_valid_scaled, X_test_scaled

Data transformation

In [149]:
df = pd.read_csv("data.csv")


df['mnth'] = df.apply(lambda row: np.sin((row['mnth']-4)*2*np.pi/12), axis=1)
df['hr'] = df.apply(lambda row: np.sin((row['hr']-11)*2*np.pi/24), axis=1)
# df["monthday"] = df.apply(lambda row: int(row["dteday"].split("-")[-1]), axis=1)
df = df.drop(['instant','dteday', 'season'], axis=1)

test_df = pd.read_csv("evaluation_data.csv")


test_df["mnth"] = test_df.apply(
    lambda row: np.sin((row["mnth"] - 4) * 2 * np.pi / 12), axis=1
)
test_df["hr"] = test_df.apply(
    lambda row: np.sin((row["hr"] - 11) * 2 * np.pi / 24), axis=1
)
# df["monthday"] = df.apply(lambda row: int(row["dteday"].split("-")[-1]), axis=1)
test_df = test_df.drop(["dteday", "season"], axis=1)
test_df

,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed
0,0,-1.000000,-2.588190e-01,0,4,1,1,0.26,0.2273,0.56,0.3881
1,0,-1.000000,-5.000000e-01,0,4,1,1,0.26,0.2727,0.56,0.0000
2,0,-1.000000,-7.071068e-01,0,4,1,1,0.26,0.2727,0.56,0.0000
3,0,-1.000000,-8.660254e-01,0,4,1,1,0.26,0.2576,0.56,0.1642
4,0,-1.000000,-9.659258e-01,0,4,1,1,0.26,0.2576,0.56,0.1642
...,...,...,...,...,...,...,...,...,...,...,...
6488,1,-0.866025,8.660254e-01,0,1,1,2,0.26,0.2576,0.60,0.1642
6489,1,-0.866025,7.071068e-01,0,1,1,2,0.26,0.2576,0.60,0.1642
6490,1,-0.866025,5.000000e-01,0,1,1,1,0.26,0.2576,0.60,0.1642
6491,1,-0.866025,2.588190e-01,0,1,1,1,0.26,0.2727,0.56,0.1343


Creating train valid and test dataset and standarizing data (only last column as label)

In [150]:
train = df.sample(frac=0.99, random_state=200)  # random state is a seed value
_rest = df.drop(train.index)
valid = _rest.sample(frac=0.5, random_state=200)  # random state is a seed value
test = test_df.sample(frac=0.99, random_state=200)
print(f"{train.shape=}")
print(f"{valid.shape=}")
print(f"{test.shape=}")

X_train = train.values[:, :-3]
y_train = train.values[:, -1]

X_valid = valid.values[:, :-3]
y_valid = valid.values[:, -1]

X_test = test.values[:, :]
y_test = test.values[:, -1]

X_train, X_valid, X_test = standarized_data(X_train, X_valid, X_test)
print(f"{X_train.shape=}")
print(f"{X_valid.shape=}")
print(f"{X_test.shape=}")
print(f"{y_train.shape=}")
print(f"{y_valid.shape=}")
print(f"{y_test.shape=}")

train.shape=(10777, 14)
valid.shape=(54, 14)
test.shape=(6428, 11)
X_train.shape=(10777, 11)
X_valid.shape=(54, 11)
X_test.shape=(6428, 11)
y_train.shape=(10777,)
y_valid.shape=(54,)
y_test.shape=(6428,)


In [151]:
train_dataset = data.TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train).long()
)
valid_dataset = data.TensorDataset(
    torch.from_numpy(X_valid).float(),
    torch.from_numpy(y_valid).long()
)
test_dataset = data.TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test).long()
)

Datasets are now ready
  
---

In [152]:
def create_model(model_class, num_inputs, num_hidden, num_outputs):
    return model_class(num_inputs=num_inputs, num_hidden=num_hidden, num_outputs=num_outputs)

In [153]:
class RegressionModel(nn.Module):

    def __init__(self, num_inputs, num_hidden, num_outputs=1):
        super().__init__()
        # Initialize the modules we need to build the network
        self.linear1 = nn.Linear(num_inputs, num_hidden)
        self.act_fn = nn.ReLU()
        # self.norm2 = nn.BatchNorm1d(num_hidden)
        # self.linear2 = nn.Linear(num_hidden, num_hidden)
        self.linear3 = nn.Linear(num_hidden, num_outputs)

        # nn.init.xavier_uniform_(self.linear1.weight)
        # nn.init.zeros_(self.linear1.bias)
        # nn.init.xavier_uniform_(self.linear2.weight)
        # nn.init.zeros_(self.linear2.bias)


    def forward(self, x):
        # Perform the calculation of the model to determine the prediction
        x = self.linear1(x)
        x = self.act_fn(x)
        # x = self.norm2(x)
        # x = self.linear2(x)
        # x = self.act_fn(x)
        x = self.linear3(x)
        return x

In [154]:
def train_model(model, train_dataset, batch_size, num_epochs, loss_module, optimizer, lr, verbose=0):
    train_data_loader = data.DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True
    )

    optimizer = optimizer(model.parameters(), lr=lr)
    model.train()

    for epoch in range(num_epochs):
        for data_input, data_label in train_data_loader:
            # print(data_input)
            pred = model(data_input)
            # print(pred)
            # print(pred.shape)
            # pred = pred.squeeze(dim=1)
            # print(pred.shape)
            # print(data_label)
            # print(data_label.shape)
            data_label = data_label.view(-1, 1).float()

            # print(pred.shape)
            # print(data_label.shape)

            loss = loss_module(pred, data_label)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        if verbose > 0 and epoch % (num_epochs//10) == 0:
            print(f"Epoch: {epoch}, loss: {loss.item():.3}")

    return model

In [155]:
def evaluate_model(model, test_dataset, loss_module):
    test_data_loader = data.DataLoader(
        test_dataset, batch_size=1, shuffle=False, drop_last=False
    )

    model.eval()  # Set model to eval mode
    loss = 0.0
    size = 0

    with torch.no_grad():  # Deactivate gradients for the following code
        for data_inputs, data_labels in test_data_loader:
            size += 1
            # Determine prediction of model on dev set
            # data_inputs, data_labels = data_inputs.to(device), data_labels.to(device)
            preds = model(data_inputs)
            # print(preds.min().item(), preds.max().item())
            # print(preds.shape)
            # preds = preds.squeeze(dim=1)
            # pred_class = preds.squeeze(dim=1).round().float()
            # print(pred_class.shape)
            # pred_class = torch.argmax(preds, dim=1)
            # print(pred_class.shape)
            # print(data_labels.shape)
            # true_class = torch.argmax(data_labels, dim=1)
            # print(true_class.shape)
            data_labels = data_labels.view(-1, 1).float()

            # Keep records of predictions for the accuracy metric (true_preds=TP+TN, num_preds=TP+TN+FP+FN)
            loss += loss_module(preds, data_labels)

    total_loss = loss / size
    print(f" RMSLE error of the model: {total_loss:4.3f}")

In [156]:
INPUT_SIZE = next(iter(train_dataset))[0].shape[0]
BATCH_SIZE = 128
LEARNING_RATE_OPTIMIZER = 0.1
OPTIMIZER = torch.optim.Adam
LOSS_MODULE = RMSLELoss()
NUM_EPOCHS = 200
OUTPUT_SIZE = 1
HIDDEN_SIZE = 64

# for hs in [16, 32, 64, 128, 256]:
#     print(f'{hs=}')
model = RegressionModel(num_inputs=INPUT_SIZE, num_hidden=HIDDEN_SIZE, num_outputs=OUTPUT_SIZE)
train_model(model, train_dataset, batch_size=BATCH_SIZE, num_epochs=NUM_EPOCHS, loss_module=LOSS_MODULE, optimizer=OPTIMIZER, lr=LEARNING_RATE_OPTIMIZER, verbose=1)
evaluate_model(model, valid_dataset, LOSS_MODULE)

Epoch: 0, loss: 1.3
Epoch: 20, loss: 1.28
Epoch: 40, loss: 0.993
Epoch: 60, loss: 1.19
Epoch: 80, loss: 0.977
Epoch: 100, loss: 0.97
Epoch: 120, loss: 1.26
Epoch: 140, loss: 0.999
Epoch: 160, loss: 1.29
Epoch: 180, loss: 0.583
 RMSLE error of the model: 0.772


In [157]:
def generate_predictions(model, dataset, output_file="predictions.csv"):
    loader = data.DataLoader(dataset, batch_size=1, shuffle=False, drop_last=False)

    model.eval()
    preds_list = []

    with torch.no_grad():
        for data_inputs, _ in loader:
            preds = model(data_inputs)

            preds = preds.item()
            preds_list.append(preds)

    pd.Series(preds_list).to_csv(output_file, index=False, header=False)

    print(f"Saved {len(preds_list)} predictions to {output_file}")

In [140]:
model = RegressionModel(num_inputs=INPUT_SIZE, num_hidden=HIDDEN_SIZE, num_outputs=OUTPUT_SIZE)
train_model(
    model,
    train_dataset,
    batch_size=BATCH_SIZE,
    num_epochs=NUM_EPOCHS,
    loss_module=LOSS_MODULE,
    optimizer=OPTIMIZER,
    lr=LEARNING_RATE_OPTIMIZER,
    verbose=0,
)

RegressionModel(
  (linear1): Linear(in_features=11, out_features=64, bias=True)
  (act_fn): ReLU()
  (linear3): Linear(in_features=64, out_features=1, bias=True)
)

In [158]:
generate_predictions(model, train_dataset)

Saved 10777 predictions to predictions.csv
